In [2]:
from PIL import Image
import httpx
from io import BytesIO
from transformers import AutoProcessor, AutoModel
import torch

model = AutoModel.from_pretrained("google/siglip2-base-patch16-224")
processor = AutoProcessor.from_pretrained("google/siglip2-base-patch16-224")

c:\Dev\SigLIP2_compression\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 408/408 [00:00<00:00, 7322.82it/s]


In [ ]:

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
with httpx.stream("GET", url) as response:
    image = Image.open(BytesIO(response.read()))

texts = ["a photo of 2 cats", "a photo of 2 dogs"]
# important: we pass `padding=max_length` since the model was trained with this
inputs = processor(text=texts, images=image, padding="max_length", return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits_per_image = outputs.logits_per_image
probs = torch.sigmoid(logits_per_image) # these are the probabilities
print(f"{probs[0][0]:.1%} that image 0 is '{texts[0]}'")

In [8]:
model

SiglipModel(
  (text_model): SiglipTextModel(
    (embeddings): SiglipTextEmbeddings(
      (token_embedding): Embedding(256000, 768)
      (position_embedding): Embedding(64, 768)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-11): 12 x SiglipEncoderLayer(
          (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attn): SiglipAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): SiglipMLP(
            (activation_fn): GELUTanh()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768,

In [10]:
# Получаем параметры vision encoder
vision_encoder = model.vision_model
# Получаем параметры text encoder
text_encoder = model.text_model

# Функция для подсчета параметров
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# Количество параметров в vision и text энкодерах
vision_params = count_parameters(vision_encoder)
text_params = count_parameters(text_encoder)

# Выводим количество параметров
print(f"Количество параметров в vision энкодере: {vision_params}")
print(f"Количество параметров в text энкодере: {text_params}")

# Сравниваем
if vision_params > text_params:
    print("Vision энкодер имеет больше параметров, чем текстовый энкодер.")
else:
    print("Текстовый энкодер имеет больше параметров, чем vision энкодер.")

Количество параметров в vision энкодере: 92884224
Количество параметров в text энкодере: 282303744
Текстовый энкодер имеет больше параметров, чем vision энкодер.


In [ ]:
import pandas as pd
from PIL import Image
import os
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor

In [111]:


class Flickr30kDataset(Dataset):
    def __init__(self, csv_file, image_dir, processor, split="test"):
        self.data = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.processor = processor
        self.split = split
        self.data = self.data[self.data['split'] == self.split]
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Загрузка изображения
        img_path = os.path.join(self.image_dir, row['filename'])
        image = Image.open(img_path).convert("RGB")
        
        # Получение описаний (5 описаний на одно изображение)
        captions = row['raw'][2:-2].replace('"', '').split(', ')  # Обработка списка описаний
        
        # Применение processor к изображениям и тексту
        inputs = self.processor(text=captions, images=image, return_tensors="pt", padding="max_length", max_length=64)
        # print(list(inputs.keys()))
        # print(inputs['pixel_values'].shape)
        # print(inputs['input_ids'].shape)
        all_input_ids = []
        for caption in captions:
            text_inputs = self.processor(
                text=caption, 
                images=image, 
                return_tensors="pt", 
                padding="max_length", 
                max_length=64,
                truncation=True
            )
            all_input_ids.append(text_inputs['input_ids'].squeeze(0))
        
        
        # Стекуем все 5 описаний
        input_ids = torch.stack(all_input_ids[:5])

        # print(inputs['pixel_values'][0].shape, input_ids.shape, len(captions)) 
        
        return {
            "image": inputs['pixel_values'][0],  # Только первый элемент из batch
            # "captions": captions,
            "input_ids": input_ids,  # Для текста
            # "attention_mask": inputs['attention_mask'][0]
        }

# Параметры пути
csv_file = "D:/flickr30k/flickr_annotations_30k.csv"
image_dir = "D:/flickr30k/flickr30k-images/flickr30k-images/"

# Загрузка тестовой выборки
processor = AutoProcessor.from_pretrained("google/siglip2-base-patch16-224")
dataset = Flickr30kDataset(csv_file, image_dir, processor, split="test")
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

In [112]:
print(len(dataloader))

32


In [ ]:
import torch
import torch.nn.functional as F

# Функция для вычисления cosine similarity
def cosine_similarity(a, b):
    return F.cosine_similarity(a, b, dim=1)

for batch in dataloader:
    images = batch["image"]
    # print(images.shape)
    captions = batch["input_ids"]
    # print(len(captions))
    # print(captions.shape)
    image_embedding = model.get_image_features(images).pooler_output
    text_embedding = model.get_text_features(captions).pooler_output

    # Normalize embeddings for cosine similarity
    image_embedding = image_embedding / image_embedding.norm(p=2, dim=-1, keepdim=True)
    text_embedding = text_embedding / text_embedding.norm(p=2, dim=-1, keepdim=True)

    # print(image_embedding.shape, text_embedding.shape)

    for i in range(5):
        # print(image_embedding.shape, text_embedding[i::5].shape)
        sim = cosine_similarity(image_embedding, text_embedding[i::5])
        print(f"Cosine similarity: {sim}")

In [114]:
import numpy as np
from tqdm import tqdm
text_embeddings = []
image_embeddings = []

recall_count = 0
total = 0
for batch in tqdm(dataloader):
    images = batch["image"]
    captions = batch["input_ids"]
    # Получаем эмбеддинги
    image_embedding = model.get_image_features(images).pooler_output
    text_embedding = model.get_text_features(captions).pooler_output
    # Normalize embeddings for cosine similarity
    image_embedding = image_embedding / image_embedding.norm(p=2, dim=-1, keepdim=True)
    text_embedding = text_embedding / text_embedding.norm(p=2, dim=-1, keepdim=True)
    
    image_embeddings.append(image_embedding)
    # Сохраняем эмбеддинги
    for i in range(5):
        text_embeddings.append(text_embedding[i::5])
    
# Объединяем все эмбеддинги
all_image_embeds = torch.cat(text_embeddings, dim=0)
all_text_embeds = torch.cat(image_embeddings, dim=0)

# Вычисляем recall@1 для text-to-image
correct = 0
total = len(all_text_embeds)

# Для каждого текстового запроса
for i in tqdm(range(total)):
    # Вычисляем сходство с каждым изображением
    text_embed = all_text_embeds[i:i+1]  # [1, dim]
    similarities = torch.matmul(text_embed, all_image_embeds.T)  # [1, num_images]
    
    # Находим индекс наиболее похожего изображения
    top1_idx = similarities.argmax(dim=1).item()
    
    # Определяем, какой индекс изображения соответствует этому тексту
    # Каждые 5 текстов соответствуют одному изображению
    true_image_idx = i // 5
    
    if top1_idx == true_image_idx:
        correct += 1


recall_at_1 = correct / total
print(f"Recall@1 (text to image retrieval): {recall_at_1}")

 19%|█▉        | 6/32 [06:06<26:28, 61.10s/it]


KeyboardInterrupt: 

In [ ]:
import numpy as np
from tqdm import tqdm
text_embeddings = []
image_embeddings = []

recall_count = 0
total = 0
# for batch in tqdm(dataloader):
#     images = batch["image"]
#     captions = batch["input_ids"]
#     # Получаем эмбеддинги
#     image_embedding = model.get_image_features(images).pooler_output
#     text_embedding = model.get_text_features(captions).pooler_output
#     # Normalize embeddings for cosine similarity
#     image_embedding = image_embedding / image_embedding.norm(p=2, dim=-1, keepdim=True)
#     text_embedding = text_embedding / text_embedding.norm(p=2, dim=-1, keepdim=True)
    
#     # Сохраняем эмбеддинги
#     for i in range(5):
#         text_embeddings.append(text_embedding[i::5])
#         image_embeddings.append(image_embedding)
    
# # Объединяем все эмбеддинги
# all_image_embeds = torch.cat(text_embeddings, dim=0)
# all_text_embeds = torch.cat(image_embeddings, dim=0)

# Вычисляем recall@1 для text-to-image
correct = 0
total = len(all_image_embeds) / 5

# Для каждого текстового запроса
for i, image_embed in tqdm(enumerate(all_image_embeds)):
    # Вычисляем сходство с каждым изображением
    similarities = torch.matmul(image_embed.unsqueeze(0), all_text_embeds.T)  # [1, num_images]
    
    # Находим индекс наиболее похожего изображения
    top1_idx = similarities.argmax(dim=1).item()
    
    # Определяем, какой индекс изображения соответствует этому тексту
    # Каждые 5 текстов соответствуют одному изображению
    true_text_indices = range(i * 5, (i + 1) * 5)
        
    if top1_idx in true_text_indices:
        correct += 1


recall_at_1 = correct / total
print(f"Recall@1 (image to text retrieval): {recall_at_1}")